# Wall segmentation (UNet) — FloorPlanCAD, Kaggle background training

Trains a binary wall vs. not-wall UNet on the original FloorPlanCAD SVG
release (the `catwhisker/floorplancad-dataset` `*.tar.xz` archives).
Auto-resumes across Kaggle's ~12 h `Save & Run All` sessions.

## One-time setup
1. **Accelerator → GPU T4 x2**, **Internet → On**.
2. **Add Input** → attach **`catwhisker/floorplancad-dataset`** (the 5 GB SVG tarballs).
3. Create an empty checkpoint dataset once, e.g. **`wallseg-ckpt`**.
4. **Add-ons → Secrets** → `KAGGLE_USERNAME`, `KAGGLE_KEY`.

## Order of operations
Run cells **1–3 first**, paste the probe output to identify the wall
`semantic-id`, set `WALL_IDS` in cell 4, then run the rest.

## 1. Environment + Kaggle API creds

In [ ]:
!pip install -q segmentation-models-pytorch==0.3.4 PyMuPDF
import os
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    os.environ['KAGGLE_USERNAME'] = s.get_secret('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = s.get_secret('KAGGLE_KEY')
    print('Kaggle API creds loaded.')
except Exception as e:
    print('No Secrets — checkpoint upload disabled:', e)
!nvidia-smi | head -15

## 2. Get the code

In [ ]:
import os, sys
REPO = '/kaggle/working/training-archnetv2'
if os.path.isdir(REPO):
    !cd {REPO} && git pull
else:
    !git clone https://github.com/miladmirzazadeh/training-archnetv2.git {REPO}
os.chdir(REPO); sys.path.insert(0, REPO)
print('cwd:', os.getcwd())

## 3. Probe the SVG schema — RUN FIRST
Extracts a few SVGs and prints the `semantic-id` histogram + stroke
colors. **Walls are drawn in dark red** — find that id and paste the
output back. SRC is auto-detected as the folder holding the `*.tar.xz`.

In [ ]:
import glob, os
tars = glob.glob('/kaggle/input/**/*.tar.xz', recursive=True)
assert tars, 'Attach catwhisker/floorplancad-dataset (the *.tar.xz archives).'
SRC = os.path.dirname(tars[0])
print('SRC:', SRC, '| tars:', [os.path.basename(t) for t in tars])
!python -m wallseg.prepare_floorplancad_walls probe --src "{SRC}" --work /kaggle/working/fpcad --sample 8

## 4. Convert SVG → wall masks
**Set `WALL_IDS`** to the dark-red id(s) from the probe (comma-separated).
Check the printed *mean wall pixel fraction* (~0.02–0.15 is healthy; ~0
means wrong ids).

In [ ]:
WALL_IDS = '1,2'   # <-- EDIT from probe output (dark-red wall semantic-id)
!python -m wallseg.prepare_floorplancad_walls convert \
    --src "{SRC}" --work /kaggle/working/fpcad --dst /kaggle/working/walls_seg \
    --wall-ids {WALL_IDS} --imgsz 640 --line-w 4

## 5. Train the UNet (auto-resume + time budget)

In [ ]:
CKPT = 'YOUR_USERNAME/wallseg-ckpt'   # <-- edit
!python -m wallseg.train \
    --data /kaggle/working/walls_seg \
    --ckpt-dataset {CKPT} \
    --time-budget 11.0 --epochs 100 --batch 8 --imgsz 640 --device 0

## 6. Check weights

In [ ]:
!ls -la /kaggle/working/runs/wallseg/wallseg_unet/ 2>/dev/null || echo 'no weights yet'